Task 1 – Environment Setup and Version Check  (15 marks)


In [ ]:
import sys, numpy as np, torch
import tensorflow as tf

print('Python :', sys.version.split()[0])
print('NumPy  :', np.__version__)
print('PyTorch:', torch.__version__)
print('TF     :', tf.__version__)

Python : 3.13.15
NumPy  : 2.1.3
PyTorch: 2.11.0+cpu
TF     : 2.20.0


Task 2 – GPU Verification and CPU vs GPU Benchmark  (20 marks)

In [ ]:
import torch, time
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name  :', torch.cuda.get_device_name(0))

def benchmark(device, n=4000, repeats=3):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(repeats):
        c = a @ b
    if device == 'cuda': torch.cuda.synchronize()
    return (time.time() - t0) / repeats

t_cpu = benchmark('cpu')
print(f'CPU: {t_cpu:.4f} s')
if torch.cuda.is_available():
    t_gpu = benchmark('cuda')
    print(f'GPU: {t_gpu:.4f} s   speed-up: {t_cpu/t_gpu:.1f}x')

CUDA available: True
Device name  : Tesla T4
CPU: 1.0148 s
GPU: 0.0845 s   speed-up: 12.0x


Task 3 – Vectorization: Removing Loops  (20 marks)

In [ ]:
import numpy as np, time
np.random.seed(0)
a = np.random.rand(1_000_000)
b = np.random.rand(1_000_000)

t0 = time.time()
total = 0.0
for i in range(len(a)):
    total += a[i] * b[i]
t_loop = time.time() - t0

t0 = time.time()
total_vec = np.dot(a, b)
t_vec = time.time() - t0

print(f'loop      : {total:.6f}  in {t_loop*1000:.1f} ms')
print(f'vectorized: {total_vec:.6f}  in {t_vec*1000:.3f} ms')
print(f'speed-up  : {t_loop/t_vec:.0f}x')

loop      : 250116.560737  in 401.5 ms
vectorized: 250116.560737  in 1.316 ms
speed-up  : 305x


Task 4 – Broadcasting  (15 marks)

In [ ]:
import numpy as np
A = np.array([[56.0, 0.0, 4.4, 68.0],
              [ 1.2, 104.0, 52.0, 8.0],
              [ 1.8, 135.0, 99.0, 0.9]])

col_sum = A.sum(axis=0)          # shape (4,)
percent = 100 * A / col_sum      # (3,4) / (4,) -> broadcast
print(percent)

[[94.91525424  0.          2.83140283 88.42652796]
 [ 2.03389831 43.51464435 33.46203346 10.40312094]
 [ 3.05084746 56.48535565 63.70656371  1.17035111]]


Task 5 – Tensors and Device Transfer  (10 marks)

In [ ]:
import torch, numpy as np

t1 = torch.zeros(2, 3)
t2 = torch.ones(2, 3)
t3 = torch.randn(2, 3)
print(t3, t3.shape, t3.dtype, t3.device)

n = np.array([[1., 2.], [3., 4.]])   # NumPy  -> tensor
t = torch.from_numpy(n)
back = t.numpy()                      # tensor -> NumPy

device = 'cuda' if torch.cuda.is_available() else 'cpu'
t_gpu = t3.to(device)
print('on device:', t_gpu.device)

tensor([[ 0.3740, -1.3969,  0.0842],
        [-0.2428,  0.0904,  0.4134]]) torch.Size([2, 3]) torch.float32 cpu
on device: cuda:0


Task 6 – Automatic Differentiation (Autograd)  (15 marks)

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 5*x + 2          # dy/dx = 2x + 5  -> 11 at x = 3
y.backward()
print('dy/dx at x=3 :', x.grad.item())

# Vector input, scalar output
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
z = (w**2).sum()            # dz/dw = 2w
z.backward()
print('dz/dw :', w.grad)

dy/dx at x=3 : 11.0
dz/dw : tensor([2., 4., 6.])
